# Checking Lemma 1 by Counting People

**Paper:** Wang, Liu & Levy (2021). *Fair Classification with Group-Dependent Label Noise.* FAccT '21.

## What this notebook does

Lemma 1 claims you can recover a classifier's **true** true-positive rate (TPR) from rates measured on **noisy** labels:

> **TPR (true) = (1 − ε⁺) × TPR (noisy) + ε⁺ × FPR (noisy)**

where **ε⁺** is the chance that a truly positive person got the wrong label (−1).

Rather than check this with algebra, we'll check it the way an auditor would:

1. Build the paper's own Example 2 as a table of **individual people**.
2. Give each person a true label and a noisy (recorded) label.
3. Run the paper's classifier on them.
4. **Count** the four rates directly.
5. Plug the counts into Lemma 1 and see whether its prediction matches what we counted.

**No math background is needed.** Every number in this notebook comes from counting rows in a table. If a formula disagrees with a count, the formula is wrong, because the count is the ground truth.

### Map of the notebook

| Part | Question it answers |
|---|---|
| 1 | Can we rebuild the paper's Example 2 exactly? |
| 2 | What are the true and noisy rates, by counting? |
| 3 | Does Lemma 1's prediction match the count? |
| 4 | *Where* does Lemma 1 go wrong? (the key part) |
| 5 | Sanity check with a perfect classifier |
| 6 | What is the correct relationship? |
| 7 | Stress test on every possible classifier |
| 8 | Does the paper's Example 1 conclusion still hold? |
| 9 | Plain-language summary |

---
## Setup

Two standard libraries: **pandas** for tables and **matplotlib** for one chart.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.precision", 3)

---
## Part 1: Rebuild Group B from Example 2

### What we're building

Group B in Example 2 has 600 people spread over four "cells," each defined by two yes/no features (x1, x2). The paper's table gives, for each cell, the person's **true label** and how many people ended up **recorded** as +1 versus −1.

| Cell (x1, x2) | True label | Recorded +1 | Recorded −1 | What happened |
|---|---|---|---|---|
| (0,0) | −1 | 75 | 225 | 75 true negatives were **wrongly recorded as +1** |
| (0,1) | +1 | 75 | 25 | 25 true positives were **wrongly recorded as −1** |
| (1,0) | +1 | 75 | 25 | 25 true positives wrongly recorded as −1 |
| (1,1) | +1 | 75 | 25 | 25 true positives wrongly recorded as −1 |

### The code below

It creates one row per person, with four columns: `x1`, `x2`, `true_label`, and `noisy_label`.

**Input:** the counts from the table above.
**Output:** a table of 600 people.

In [ ]:
def make_people(x1, x2, true_label, n_recorded_pos, n_recorded_neg):
    # Create rows for everyone in one cell.
    # n_recorded_pos people get noisy_label = +1, n_recorded_neg get -1.
    rows = []
    for _ in range(n_recorded_pos):
        rows.append({"x1": x1, "x2": x2, "true_label": true_label, "noisy_label": +1})
    for _ in range(n_recorded_neg):
        rows.append({"x1": x1, "x2": x2, "true_label": true_label, "noisy_label": -1})
    return rows

# Group B, straight from the paper's Example 2 table
group_B = pd.DataFrame(
      make_people(0, 0, true_label=-1, n_recorded_pos=75, n_recorded_neg=225)
    + make_people(0, 1, true_label=+1, n_recorded_pos=75, n_recorded_neg=25)
    + make_people(1, 0, true_label=+1, n_recorded_pos=75, n_recorded_neg=25)
    + make_people(1, 1, true_label=+1, n_recorded_pos=75, n_recorded_neg=25)
)

print("Total people in Group B:", len(group_B))
group_B.head()

### Check: does our table match the paper?

We summarize the 600 rows back into the paper's format. The numbers should match the table above.

In [ ]:
check = pd.crosstab(
    [group_B["x1"], group_B["x2"], group_B["true_label"]],
    group_B["noisy_label"],
).rename(columns={1: "recorded +1", -1: "recorded -1"})[["recorded +1", "recorded -1"]]
check

**✅ Match.** We now have Group B exactly as the paper describes it.

---
## Part 2: Run the classifier and count the four rates

### The classifier

The paper's "fair" classifier from Example 2, `f*_fair`, predicts **+1 when x2 = 1** and −1 otherwise.

### The four rates, in plain words

All four answer the same question: **"How often does the classifier say +1 to this group of people?"** They differ only in *which* group of people.

| Rate | Asks how often the classifier says +1 to... | Uses which labels? |
|---|---|---|
| **True TPR** | people who are *truly* positive | true (what we really care about) |
| **True FPR** | people who are *truly* negative | true |
| **Noisy TPR** | people *recorded* as positive | noisy (what you can actually measure) |
| **Noisy FPR** | people *recorded* as negative | noisy |

In real life you can only compute the noisy rates, because you don't know the true labels. Lemma 1's whole purpose is to recover the true rates from the noisy ones.

**Input:** the 600-person table.
**Output:** the four rates, each computed by counting.

In [ ]:
# The paper's f*_fair: predict +1 when x2 == 1
def f_fair(df):
    return df["x2"].map({1: +1, 0: -1})

group_B["prediction"] = f_fair(group_B)

def rate_of_plus1(people):
    # Fraction of these people the classifier says +1 to
    return (people["prediction"] == +1).mean()

def four_rates(df):
    return {
        "true TPR":  rate_of_plus1(df[df["true_label"]  == +1]),
        "true FPR":  rate_of_plus1(df[df["true_label"]  == -1]),
        "noisy TPR": rate_of_plus1(df[df["noisy_label"] == +1]),
        "noisy FPR": rate_of_plus1(df[df["noisy_label"] == -1]),
    }

rates = four_rates(group_B)
for name, value in rates.items():
    print(f"{name:10s} = {value:.3f}")

### Reading the output

- **True TPR = 0.667.** Of the 300 truly positive people, the classifier says +1 to 200. This matches the paper's text ("2/3").
- **Noisy TPR = 0.500.** Of the 300 people *recorded* positive, it says +1 to 150.

These differ because the "recorded positive" group contains 75 impostors, true negatives who were mislabeled. That turns out to be the whole story.

### Noise rates, also by counting

Lemma 1 needs **ε⁺**: the fraction of truly positive people who were recorded as −1.

In [ ]:
true_pos = group_B[group_B["true_label"] == +1]
true_neg = group_B[group_B["true_label"] == -1]

eps_plus  = (true_pos["noisy_label"] == -1).mean()   # true positives recorded as -1
eps_minus = (true_neg["noisy_label"] == +1).mean()   # true negatives recorded as +1

print(f"eps_plus  (positives mislabeled) = {eps_plus:.3f}")
print(f"eps_minus (negatives mislabeled) = {eps_minus:.3f}")

Both are 0.25, which matches the paper's statement that "one quarter of group B's labels are incorrect."

---
## Part 3: Test Lemma 1

### What we're doing

Plug the counted noisy rates and ε⁺ into Lemma 1:

> **predicted true TPR = (1 − ε⁺) × noisy TPR + ε⁺ × noisy FPR**

Then compare with the true TPR we **counted** in Part 2.

In [ ]:
lemma1_prediction = (1 - eps_plus) * rates["noisy TPR"] + eps_plus * rates["noisy FPR"]

print(f"Lemma 1 predicts true TPR = {lemma1_prediction:.3f}")
print(f"Counted true TPR          = {rates['true TPR']:.3f}")
print(f"Difference                = {rates['true TPR'] - lemma1_prediction:+.3f}")

### ❌ They don't match

Lemma 1 predicts 0.417. Counting gives 0.667. The formula misses by 25 percentage points, on the paper's own example.

Counting can't be wrong, so the formula must be. Part 4 shows exactly why.

---
## Part 4: Where Lemma 1 goes wrong

### The idea behind Lemma 1

The authors split the **truly positive** people into two piles:

- **Pile 1:** truly positive and **correctly** recorded as +1
- **Pile 2:** truly positive but **wrongly** recorded as −1

So far, so good. Then they make a substitution:

| They need... | They substitute... |
|---|---|
| How often the classifier says +1 to **Pile 1** | **Noisy TPR**, its rate among *everyone* recorded +1 |
| How often the classifier says +1 to **Pile 2** | **Noisy FPR**, its rate among *everyone* recorded −1 |

That substitution only works if the extra people in each recorded group (the impostors) get treated **exactly like** the real members. Let's check.

### The code below

We split the people **recorded +1** into real positives and impostors, and count how often the classifier says +1 to each. Then we do the same for the people **recorded −1**.

**Input:** the 600-person table with predictions.
**Output:** the classifier's +1 rate for each sub-pile.

In [ ]:
recorded_pos = group_B[group_B["noisy_label"] == +1]
recorded_neg = group_B[group_B["noisy_label"] == -1]

breakdown = pd.DataFrame([
    {"recorded as": "+1", "who": "real positives (Pile 1)",
     "count": (recorded_pos["true_label"] == +1).sum(),
     "classifier says +1 rate": rate_of_plus1(recorded_pos[recorded_pos["true_label"] == +1])},
    {"recorded as": "+1", "who": "impostors (true negatives)",
     "count": (recorded_pos["true_label"] == -1).sum(),
     "classifier says +1 rate": rate_of_plus1(recorded_pos[recorded_pos["true_label"] == -1])},
    {"recorded as": "-1", "who": "hidden positives (Pile 2)",
     "count": (recorded_neg["true_label"] == +1).sum(),
     "classifier says +1 rate": rate_of_plus1(recorded_neg[recorded_neg["true_label"] == +1])},
    {"recorded as": "-1", "who": "real negatives",
     "count": (recorded_neg["true_label"] == -1).sum(),
     "classifier says +1 rate": rate_of_plus1(recorded_neg[recorded_neg["true_label"] == -1])},
])
breakdown

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)

for ax, label, sub in [(axes[0], "+1", breakdown.iloc[0:2]), (axes[1], "-1", breakdown.iloc[2:4])]:
    bars = ax.bar(sub["who"], sub["classifier says +1 rate"], color=["#2a7ab9", "#d9822b"])
    for b, v in zip(bars, sub["classifier says +1 rate"]):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.02, f"{v:.2f}", ha="center")
    ax.set_title(f"People RECORDED as {label}")
    ax.set_ylim(0, 1)
    ax.tick_params(axis="x", labelsize=9)

axes[0].set_ylabel("How often classifier says +1")
fig.suptitle("The classifier does NOT treat impostors like real members")
plt.tight_layout()
plt.show()

### This is the error

Look at the left chart. Among people **recorded +1**:
- The classifier says +1 to **67%** of the real positives.
- The classifier says +1 to **0%** of the impostors.

It can tell them apart because it sees the features (x1, x2), and the features carry information about the true label. Lemma 1 assumes it *can't* tell them apart. The authors justify this by saying the classifier only knows what the noisy label tells it, but that overlooks the features.

**The consequence:**
- Noisy TPR (0.50) blends the real positives (0.67) with impostors (0.00). Lemma 1 uses that diluted 0.50 as if it were Pile 1's rate. The true value is 0.67.
- Noisy FPR (0.17) blends hidden positives (0.67) with real negatives (0.00). Lemma 1 uses 0.17 as if it were Pile 2's rate. The true value is 0.67.

Both substitutions **understate** the classifier's rate on true positives. That's why the prediction comes out too low.

### Put the correct numbers into the same structure

If we use the actual Pile 1 and Pile 2 rates instead of the substitutes, the structure itself works:

In [ ]:
pile1_rate = breakdown.loc[0, "classifier says +1 rate"]
pile2_rate = breakdown.loc[2, "classifier says +1 rate"]

fixed = (1 - eps_plus) * pile1_rate + eps_plus * pile2_rate
print(f"(1 - eps+) x Pile1 rate + eps+ x Pile2 rate = {fixed:.3f}")
print(f"Counted true TPR                             = {rates['true TPR']:.3f}")

**✅ Match.** So the *weights* in Lemma 1 are fine (Steps 1–2 of the derivation). The error is entirely in Step 3, swapping in noisy TPR/FPR for the pile rates.

The catch is that in real life you **can't** compute the pile rates, because they require knowing the true labels. That's why the authors wanted the substitution, and why it doesn't work.

---
## Part 5: Sanity check with a perfect classifier

A quick, intuitive test. Imagine a classifier that is **always right**: it says +1 to every true positive and −1 to every true negative.

- Its true TPR must be **100%**, by definition.
- What does Lemma 1 say?

**Input:** the same 600 people, with a perfect classifier.
**Output:** counted rates and Lemma 1's prediction.

In [ ]:
perfect = group_B.copy()
perfect["prediction"] = perfect["true_label"]     # always right

r = four_rates(perfect)
lemma1_perfect = (1 - eps_plus) * r["noisy TPR"] + eps_plus * r["noisy FPR"]

for name, value in r.items():
    print(f"{name:10s} = {value:.3f}")
print()
print(f"Lemma 1 predicts true TPR = {lemma1_perfect:.3f}")
print(f"Counted true TPR          = {r['true TPR']:.3f}")

### Reading this

A perfect classifier *looks* imperfect on noisy labels:
- **Noisy TPR = 0.75.** It correctly rejects the 75 impostors, but the noisy labels count that as a "miss."
- **Noisy FPR = 0.25.** It correctly accepts the 75 hidden positives, but the noisy labels count that as a "false alarm."

Lemma 1 takes those distorted numbers and predicts 62.5%, for a classifier that is right 100% of the time.

This is the clearest version of the error to keep in mind: **noise makes a good classifier look worse than it is, and Lemma 1 doesn't undo that.**

---
## Part 6: The relationship that does hold

### Turn it around

Instead of writing the true rate in terms of the noisy ones, write each **noisy rate** as a blend of the **true** ones. The people recorded +1 are a mix of real positives and impostors, so:

> **noisy TPR = (share of recorded-+1 who are truly +) × true TPR + (share who are truly −) × true FPR**

The weights are about **label purity**: "of the people recorded +1, what fraction really are +1?" They are *not* ε⁺.

This works because under the paper's own noise model, labels are flipped at random. A flipped true positive is therefore indistinguishable from an unflipped one, so the classifier treats them the same. Impostors, however, are true negatives, and they get treated like true negatives.

**Input:** counted true rates and label-purity shares.
**Output:** the blended prediction for the noisy rates, compared with what we counted.

In [ ]:
def purity_check(df, label):
    rates = four_rates(df)
    rec_pos = df[df["noisy_label"] == +1]
    rec_neg = df[df["noisy_label"] == -1]

    purity_pos = (rec_pos["true_label"] == +1).mean()  # share of recorded +1 truly +1
    share_neg  = (rec_neg["true_label"] == +1).mean()  # share of recorded -1 truly +1

    pred_noisy_tpr = purity_pos * rates["true TPR"] + (1 - purity_pos) * rates["true FPR"]
    pred_noisy_fpr = share_neg  * rates["true TPR"] + (1 - share_neg)  * rates["true FPR"]

    return pd.DataFrame({
        "classifier": [label, label],
        "rate": ["noisy TPR", "noisy FPR"],
        "blend of true rates": [pred_noisy_tpr, pred_noisy_fpr],
        "counted": [rates["noisy TPR"], rates["noisy FPR"]],
    })

pd.concat([purity_check(group_B, "paper's f*_fair"),
           purity_check(perfect, "perfect classifier")], ignore_index=True)

**✅ Exact match for both classifiers.** This is the relationship the paper should have used. If you want to go backwards (estimate true rates from noisy ones), you solve these two equations together. That requires the purity shares, which depend on the noise rates *and* on how common each class is.

---
## Part 7: Stress test on every possible classifier

One example could be a fluke. With four cells and two possible predictions each, there are exactly **16** possible classifiers. We test Lemma 1 and the corrected relationship on all of them.

We also record whether the classifier is **informative** (true TPR ≠ true FPR) or a **coin flip** (true TPR = true FPR, meaning it says +1 equally often to true positives and true negatives).

**Input:** all 16 classifiers on Group B.
**Output:** Lemma 1's error for each.

In [ ]:
from itertools import product

cells = [(0, 0), (0, 1), (1, 0), (1, 1)]
results = []

for preds in product([-1, +1], repeat=4):
    df = group_B.copy()
    lookup = dict(zip(cells, preds))
    df["prediction"] = [lookup[(a, b)] for a, b in zip(df["x1"], df["x2"])]
    r = four_rates(df)
    lemma1 = (1 - eps_plus) * r["noisy TPR"] + eps_plus * r["noisy FPR"]
    results.append({
        "predictions (00,01,10,11)": preds,
        "true TPR": r["true TPR"],
        "true FPR": r["true FPR"],
        "Lemma 1 predicts": lemma1,
        "Lemma 1 error": lemma1 - r["true TPR"],
        "type": "coin flip" if abs(r["true TPR"] - r["true FPR"]) < 1e-9 else "informative",
    })

stress = pd.DataFrame(results)
stress

In [ ]:
summary = stress.assign(lemma1_correct=stress["Lemma 1 error"].abs() < 1e-9) \
                .groupby("type")["lemma1_correct"].agg(["count", "sum"]) \
                .rename(columns={"count": "classifiers", "sum": "Lemma 1 correct"})
summary

### Reading the result

**Lemma 1 is correct only for coin-flip classifiers**, the ones that say +1 equally often to true positives and true negatives. For every classifier that actually discriminates, it is wrong.

That makes sense given Part 4. The substitution assumes the classifier can't tell real positives from impostors, which is exactly what a coin-flip classifier does. Any useful classifier *can* tell them apart, so the substitution breaks.

This is the point to remember: **Lemma 1 is right only for useless classifiers, which are exactly the classifiers no one cares about.**

---
## Part 8: Does Example 1's conclusion still hold?

The paper's most important practical claim (Theorem 1) is that enforcing equal TPR on noisy labels gives the clean group a **coin-flip classifier**. If that conclusion depended on Lemma 1, it would be in doubt.

We check it the same way as before: build Example 1, try all 16 classifiers, keep only those that satisfy **equal noisy TPR** between groups, and look at Group A's true rates.

**Input:** Example 1's table (Group A clean, Group B with 70% of negatives flipped).
**Output:** Group A's true TPR and FPR for every classifier that passes the fairness constraint.

In [ ]:
# Example 1, from the paper's table
ex1_A = pd.DataFrame(
      make_people(0, 0, -1, 0, 25) + make_people(0, 1, -1, 0, 25)
    + make_people(1, 0, +1, 25, 0) + make_people(1, 1, +1, 25, 0))
ex1_B = pd.DataFrame(
      make_people(0, 0, -1, 70, 30) + make_people(0, 1, -1, 70, 30)
    + make_people(1, 0, +1, 100, 0) + make_people(1, 1, +1, 100, 0))

passing = []
for preds in product([-1, +1], repeat=4):
    lookup = dict(zip(cells, preds))
    A, B = ex1_A.copy(), ex1_B.copy()
    for df in (A, B):
        df["prediction"] = [lookup[(a, b)] for a, b in zip(df["x1"], df["x2"])]
    rA, rB = four_rates(A), four_rates(B)
    if abs(rA["noisy TPR"] - rB["noisy TPR"]) < 1e-9:      # satisfies equal noisy TPR
        passing.append({
            "predictions (00,01,10,11)": preds,
            "noisy TPR (both groups)": rA["noisy TPR"],
            "Group A true TPR": rA["true TPR"],
            "Group A true FPR": rA["true FPR"],
            "Group A coin flip?": abs(rA["true TPR"] - rA["true FPR"]) < 1e-9,
        })

print(f"{len(passing)} of 16 classifiers satisfy equal noisy TPR:")
pd.DataFrame(passing)

### ✅ Example 1's conclusion stands

Every classifier that satisfies the fairness constraint has Group A's true TPR equal to its true FPR, so every one of them is a coin flip for the clean group. One of them is the paper's f*_fair, predictions (+1, −1, +1, −1).

We reached this by counting, without using Lemma 1. The paper's central warning survives: **enforcing fairness on group-dependently noisy labels can destroy the classifier for the group whose labels were fine.**

---
## Part 9: Plain-language summary

### What Lemma 1 gets wrong

1. Lemma 1 splits the true positives into those labeled correctly and those labeled wrongly. **That step is fine.**
2. It then assumes the classifier treats *everyone* with a given recorded label the same way, including impostors who carry the wrong label. **That step is wrong.**
3. A useful classifier sees the features, and the features reveal the truth, so it treats impostors differently. In Example 2 it said +1 to 67% of real positives but 0% of impostors.
4. As a result, Lemma 1 is correct only for coin-flip classifiers (Part 7). A perfect classifier gets a predicted TPR of 62.5% (Part 5).

### What survives

| Claim | Status | Shown in |
|---|---|---|
| Example 1 / Theorem 1: clean group gets a coin-flip classifier | ✅ Holds | Part 8 |
| Example 2: looks fair on noisy labels, unfair on true labels | ✅ Holds (rates counted directly) | Part 2 |
| Theorem 3's general message: unequal noise means "fair on noisy" ≠ "fair on true" | ✅ Holds in spirit | Part 6 relationship |
| Lemma 1 formula, Lemma 2, Table 4/5 correction constraints | ❌ Built on the faulty substitution | Parts 3–7 |

### A one-sentence version for a committee

> "Lemma 1 assumes that, within a recorded label, the classifier treats mislabeled and correctly labeled examples identically. That holds only for classifiers with no discriminatory power, so the correction formula is miscalibrated for any useful classifier. The paper's examples and qualitative conclusions, however, can be verified directly by counting."

### Before you cite this critique

- Re-run this notebook and change the numbers yourself (for example, try a different noise rate in Part 1). Seeing the mismatch persist is the best way to make it your own.
- Check whether the arXiv version of the paper or a later citing paper corrected Lemma 1.